In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path = './car_rental_dataset.xlsx'
df_bookings = pd.read_excel(file_path, sheet_name='Sheet1')

df_bookings['Booking_Date'] = pd.to_datetime(df_bookings['Booking_Date'], unit='ms', errors='coerce')
df_bookings['City'] = df_bookings['City'].astype('string').str.strip()
df_bookings['Car_Model'] = df_bookings['Car_Model'].astype('string').str.strip()
df_bookings['Category'] = df_bookings['Category'].astype('string').str.strip()

for col in ['Rental_Days', 'Total_Amount', 'Booking_ID', 'User_ID', 'Car_ID']:
    df_bookings[col] = pd.to_numeric(df_bookings[col], errors='coerce')

_df = df_bookings
_df['Year'] = _df['Booking_Date'].dt.year
_df['Month'] = _df['Booking_Date'].dt.to_period('M').astype(str)
_df['DayOfWeek'] = _df['Booking_Date'].dt.day_name()
_df['Week'] = _df['Booking_Date'].dt.to_period('W').astype(str)
_df['Amount_per_Day'] = _df['Total_Amount'] / _df['Rental_Days']

print(df_bookings.head())

sns.set_theme(style='whitegrid')

monthly_rev = _df.groupby('Month', as_index=False)['Total_Amount'].sum().sort_values('Month')
plt.figure(figsize=(10,4))
sns.lineplot(data=monthly_rev, x='Month', y='Total_Amount', marker='o')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Revenue Trend')
plt.tight_layout()
plt.show()

model_counts = _df['Car_Model'].value_counts().head(10).reset_index()
model_counts.columns = ['Car_Model', 'Bookings']
plt.figure(figsize=(10,4))
sns.barplot(data=model_counts, x='Bookings', y='Car_Model')
plt.title('Top 10 Car Models by Bookings')
plt.tight_layout()
plt.show()

city_rev = _df.groupby('City', as_index=False)['Total_Amount'].sum().sort_values('Total_Amount', ascending=False)
plt.figure(figsize=(8,4))
sns.barplot(data=city_rev, x='Total_Amount', y='City')
plt.title('Revenue by City')
plt.tight_layout()
plt.show()

cat_counts = _df['Category'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Bookings']
plt.figure(figsize=(7,4))
sns.barplot(data=cat_counts, x='Bookings', y='Category')
plt.title('Bookings by Category')
plt.tight_layout()
plt.show()

fact_bookings = _df.copy()

dim_cars = fact_bookings[['Car_ID', 'Car_Model', 'Category']].drop_duplicates().sort_values('Car_ID').reset_index(drop=True)

dim_users = fact_bookings[['User_ID']].drop_duplicates().sort_values('User_ID').reset_index(drop=True)

date_dim = fact_bookings[['Booking_Date']].dropna().drop_duplicates().sort_values('Booking_Date').reset_index(drop=True)
date_dim['Date'] = date_dim['Booking_Date'].dt.date
date_dim['Year'] = date_dim['Booking_Date'].dt.year
date_dim['MonthNum'] = date_dim['Booking_Date'].dt.month
date_dim['MonthName'] = date_dim['Booking_Date'].dt.month_name()
date_dim['Day'] = date_dim['Booking_Date'].dt.day
date_dim['DayOfWeek'] = date_dim['Booking_Date'].dt.day_name()
date_dim['Quarter'] = date_dim['Booking_Date'].dt.quarter

print(dim_cars.head())
print(date_dim.head())

fact_path = 'fact_bookings.csv'
dim_cars_path = 'dim_cars.csv'
dim_users_path = 'dim_users.csv'
dim_dates_path = 'dim_dates.csv'

fact_bookings.to_csv(fact_path, index=False)
dim_cars.to_csv(dim_cars_path, index=False)
dim_users.to_csv(dim_users_path, index=False)
date_dim.to_csv(dim_dates_path, index=False)

print(fact_path)
print(dim_cars_path)
print(dim_users_path)
print(dim_dates_path)